# 线性因果注意力（Linear Attention）

源码导航：[core/attention/linear_attention.py](../../../core/attention/linear_attention.py) 中的 `LinearCausalAttention`、`linear_attention_feature_map`。

Katharopoulos et al. (2020) 用正定核 $\phi$ 替代 softmax，使注意力可写为：

$$\text{Attn}(Q,K,V)_t = \frac{\phi(Q_t) \sum_{j \le t} \phi(K_j)^\top V_j}{\phi(Q_t) \sum_{j \le t} \phi(K_j)^\top \mathbf{1}}$$

默认 $\phi(x) = \text{elu}(x) + 1$。因果性通过前缀累积实现，单步推理维护 $(KV\_state, K\_sum)$ 即可。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.attention.linear_attention import LinearCausalAttention, linear_attention_feature_map

### 2. 序列级 forward

In [ ]:
torch.manual_seed(0)
B, T, C = 2, 8, 64
x = torch.randn(B, T, C)

attn = LinearCausalAttention(n_embd=C, n_head=4, attn_impl='eager')
y = attn(x)
assert y.shape == (B, T, C)
print('output:', tuple(y.shape))

### 3. 逐步递推推理（O(1) 每步状态更新）

In [ ]:
attn.eval()
kv_state = k_sum = None
outs = []
for t in range(T):
    step_out, kv_state, k_sum = attn.forward_step(x[:, t:t+1, :], kv_state, k_sum)
    outs.append(step_out)
recurrent = torch.cat(outs, dim=1)
print('recurrent vs batch max diff:', (recurrent - y).abs().max().item())

---

## 延伸阅读

- Katharopoulos et al. (2020). [arXiv:2006.16236](https://arxiv.org/abs/2006.16236)
- RetNet / Mamba：线性注意力思想的后续演进（本仓库后续可在 kv_cache 模块展开）。